# SqueakView Scientific Run Analysis

This notebook audits acquisition timing, serial/TTL alignment, detections, NvDCF tracks, and pose keypoints from the current DeepStream 9.1 schema. It can visualize either the live inference outputs or a provenance-preserving offline re-inference result while always treating the run's `raw.mp4` and `frames.csv` as ground truth.


In [ ]:
from pathlib import Path
import json
import subprocess

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from matplotlib.patches import Rectangle

plt.rcParams["figure.figsize"] = (12, 4)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

# Select a run explicitly, or leave RUN_DIR=None to use runs/.latest_run.
RUN_DIR = None
# "live", "latest_offline", or a Path to one offline_inference result directory.
INFERENCE_RESULT = "live"

HERE = Path.cwd().resolve()
REPO_ROOT = next((p for p in [HERE, *HERE.parents] if (p / "pyproject.toml").exists()), HERE)
if RUN_DIR is None:
    RUN_DIR = Path((REPO_ROOT / "runs" / ".latest_run").read_text().strip()).resolve()
else:
    RUN_DIR = Path(RUN_DIR).expanduser().resolve()

if INFERENCE_RESULT == "live":
    RESULT_DIR = RUN_DIR
elif INFERENCE_RESULT == "latest_offline":
    candidates = sorted((RUN_DIR / "offline_inference").glob("*"), key=lambda p: p.stat().st_mtime)
    if not candidates:
        raise FileNotFoundError("No offline inference result exists for this run")
    RESULT_DIR = candidates[-1]
else:
    RESULT_DIR = Path(INFERENCE_RESULT).expanduser().resolve()
ANALYSIS_DIR = RESULT_DIR

print(f"RUN_DIR:      {RUN_DIR}")
print(f"RESULT_DIR:   {RESULT_DIR}")
print(f"ANALYSIS_DIR: {ANALYSIS_DIR}")


## Load Current Tables

The alignment directory supplies the common RP2040 time base. Object, keypoint, and track tables come from the selected inference result. Offline results therefore remain separate from—and directly comparable with—the live outputs.


In [ ]:
def read_csv(path: Path) -> pd.DataFrame:
    return pd.read_csv(path) if path.exists() else pd.DataFrame()

def numeric(df: pd.DataFrame, columns) -> pd.DataFrame:
    for column in columns:
        if column in df:
            df[column] = pd.to_numeric(df[column], errors="coerce")
    return df

summary = json.loads((ANALYSIS_DIR / "alignment_summary.json").read_text())
frames = numeric(read_csv(RUN_DIR / "frames.csv"), [
    "raw_frame_index", "source_sequence_index", "camera_frame_id", "pts_ns", "gst_pts_ns",
    "duration_ns", "inference_admitted",
])
events = numeric(read_csv(RUN_DIR / "serial.csv"), ["rp2040Time", "count"])
objects = numeric(read_csv(RESULT_DIR / "objects.csv"), [
    "deepstream_frame_number", "source_sequence_index", "camera_frame_id", "camera_timestamp_ns",
    "gst_pts_ns", "class_id", "track_id", "detected_this_frame", "tracker_predicted",
    "detector_confidence", "tracker_confidence", "track_x", "track_y", "track_w", "track_h",
])
keypoints = numeric(read_csv(RESULT_DIR / "keypoints.csv"), [
    "deepstream_frame_number", "source_sequence_index", "camera_frame_id", "track_id", "class_id",
    "keypoint_index", "x_px", "y_px", "x_norm", "y_norm", "confidence", "visible",
])

offset = int(summary["frame_alignment"]["camera_frame_id_offset"])
first_rp2040_us = int(summary["frame_alignment"]["first_rp2040_time_us"])
highs = events[events["eventType"] == "CAMERA_HIGH"][["count", "rp2040Time"]].dropna().drop_duplicates("count")
frames["ttl_count"] = frames["camera_frame_id"] - offset
frames = frames.merge(highs.rename(columns={"count": "ttl_count", "rp2040Time": "frame_rp2040_us"}), on="ttl_count", how="left")
frames["frame_time_s"] = (frames["frame_rp2040_us"] - first_rp2040_us) / 1_000_000
frames["frame_pts_ns"] = frames["pts_ns"].fillna(frames.get("gst_pts_ns"))
frames["frame_pts_s"] = frames["frame_pts_ns"] / 1_000_000_000
frames["video_frame_index"] = frames.groupby("stream_id").cumcount()
frames["has_ttl"] = frames["frame_rp2040_us"].notna().astype(int)
object_counts = objects.groupby("source_sequence_index").size() if not objects.empty else pd.Series(dtype=int)
frames["detection_count"] = frames["raw_frame_index"].map(object_counts).fillna(0).astype(int)
frames["has_detection"] = (frames["detection_count"] > 0).astype(int)

events["serial_index"] = np.arange(len(events))
events["rp2040_time_us"] = events["rp2040Time"]
events["event_time_s"] = (events["rp2040_time_us"] - first_rp2040_us) / 1_000_000

frame_lookup = frames[["raw_frame_index", "camera_frame_id", "ttl_count", "frame_rp2040_us", "frame_time_s", "frame_pts_ns", "frame_pts_s"]].drop_duplicates("raw_frame_index")
if not objects.empty:
    objects = objects.merge(frame_lookup, left_on="source_sequence_index", right_on="raw_frame_index", how="left", suffixes=("", "_ledger"))
if not keypoints.empty:
    keypoints = keypoints.merge(frame_lookup, left_on="source_sequence_index", right_on="raw_frame_index", how="left", suffixes=("", "_ledger"))

detections = objects.copy()
if not detections.empty:
    detections["detection_index"] = np.arange(len(detections))
    detections["raw_frame_index"] = detections["source_sequence_index"]
    detections["detection_rp2040_us"] = detections["frame_rp2040_us"]
    detections["detection_time_s"] = detections["frame_time_s"]
    detections["raw_frame_mapping_method"] = "offline_video_ledger" if RESULT_DIR != RUN_DIR else "flir_user_meta"
    detections["raw_frame_mapping_ok"] = detections["camera_frame_id_ledger"].notna().astype(int) if "camera_frame_id_ledger" in detections else detections["camera_frame_id"].notna().astype(int)
    detections["raw_frame_mapping_pts_ns"] = detections["gst_pts_ns"]
    detections["conf"] = detections["detector_confidence"]
    detections["x"], detections["y"] = detections["track_x"], detections["track_y"]
    detections["w"], detections["h"] = detections["track_w"], detections["track_h"]
    detections["original_frame"] = detections["deepstream_frame_number"]

tracked = objects[objects["track_id"].notna() & (objects["track_id"] >= 0)].copy() if not objects.empty else pd.DataFrame()
if tracked.empty:
    tracks = pd.DataFrame()
else:
    tracks = tracked.groupby(["stream_id", "track_id", "class_id", "class_label"], as_index=False).agg(
        first_frame=("source_sequence_index", "min"), last_frame=("source_sequence_index", "max"),
        observed_frames=("detected_this_frame", "sum"), predicted_frames=("tracker_predicted", "sum"),
        total_rows=("observation_id", "size"),
    )

print(f"frames={len(frames):,} events={len(events):,} detections={len(detections):,}")
print(f"objects={len(objects):,} keypoints={len(keypoints):,} tracks={len(tracks):,}")
display(frames.head(3), objects.head(3), keypoints.head(3), tracks.head(3))


## Acquisition and Alignment Health

Inspect this before interpreting behavior. Ground-truth video/frame mismatches, frame gaps, missing TTL pairs, mapping fallbacks, or clock-tolerance failures must be resolved or explicitly qualified.


In [ ]:
display(pd.Series(summary.get("counts", {}), name="value").to_frame())
display(pd.Series(summary.get("validation", {}), name="value").to_frame())
display(pd.Series(summary.get("frame_identity", {}), name="value").to_frame())

checks = {
    "frame_gaps_detected": summary.get("counts", {}).get("frame_gaps_detected"),
    "frames_missing_ttl": summary.get("counts", {}).get("frames_missing_ttl"),
    "drop_events": summary.get("counts", {}).get("drop_events"),
    "object_mapping_failed_rows": summary.get("validation", {}).get("object_mapping_failed_rows"),
    "object_mapping_fallback_rows": summary.get("validation", {}).get("object_mapping_fallback_rows"),
    "object_pts_mismatch_count": summary.get("validation", {}).get("object_pts_mismatch_count"),
}
for name, value in checks.items():
    if value not in (0, None):
        print(f"WARNING: {name}={value}")

if RESULT_DIR != RUN_DIR:
    offline_manifest = json.loads((RESULT_DIR / "offline_manifest.json").read_text())
    display(pd.Series({
        "status": offline_manifest.get("status"),
        "decoded_frames": offline_manifest.get("decoded_frames"),
        "expected_frames": offline_manifest.get("source", {}).get("expected_frames"),
        "model": offline_manifest.get("model_package", {}).get("name"),
        "video_sha256": offline_manifest.get("source", {}).get("video_sha256"),
    }, name="offline provenance").to_frame())
else:
    admission_path = RUN_DIR / "inference_admission.json"
    if admission_path.exists():
        display(pd.Series(json.loads(admission_path.read_text()), name="live admission").to_frame())


## Frame Timing

RP2040 `CAMERA_HIGH` is the acquisition reference. GStreamer PTS is shown as an independent recording-path interval check; target FPS is read from the run manifest rather than hard-coded.


In [ ]:
frames = frames.sort_values("camera_frame_id").reset_index(drop=True)
frames["ttl_interval_ms"] = frames["frame_rp2040_us"].diff() / 1_000.0
frames["pts_interval_ms"] = frames["frame_pts_ns"].diff() / 1_000_000.0
run_manifest = json.loads((RUN_DIR / "run_manifest.json").read_text())
fps = float(run_manifest.get("capture", {}).get("fps", 30))
expected_ms = 1_000.0 / fps

fig, axes = plt.subplots(2, 1, figsize=(14, 7))
axes[0].plot(frames["camera_frame_id"], frames["ttl_interval_ms"], ".-", ms=3, lw=.8, label="RP2040 TTL")
axes[0].plot(frames["camera_frame_id"], frames["pts_interval_ms"], ".-", ms=3, lw=.8, label="GStreamer PTS")
axes[0].axhline(expected_ms, color="black", ls="--", lw=1, label=f"{fps:g} fps target")
axes[0].set(xlabel="FLIR camera_frame_id", ylabel="interval (ms)", title="Frame-to-frame interval")
axes[0].legend()
sns.histplot(frames["ttl_interval_ms"].dropna(), bins=40, ax=axes[1], label="RP2040", color="tab:blue")
sns.histplot(frames["pts_interval_ms"].dropna(), bins=40, ax=axes[1], label="PTS", color="tab:orange", alpha=.5)
axes[1].axvline(expected_ms, color="black", ls="--", lw=1)
axes[1].set(xlabel="interval (ms)", title="Interval distributions")
axes[1].legend()
plt.tight_layout()
display(frames[["ttl_interval_ms", "pts_interval_ms"]].describe(percentiles=[.01, .05, .5, .95, .99]))


## Inference Coverage and Serial Events

Detection coverage is aligned to every recorded frame. Tracker-predicted object rows are intentionally distinguished from detector observations.


In [ ]:
non_camera = events[~events["eventType"].isin(["CAMERA_HIGH", "CAMERA_LOW"])].dropna(subset=["event_time_s"])
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(frames["frame_time_s"], frames["detection_count"], lw=1.2, label="detection rows/frame")
if not detections.empty:
    ax.scatter(detections["detection_time_s"], detections["conf"], s=10, alpha=.4, label="detector confidence")
for _, row in non_camera.iterrows():
    ax.axvline(row["event_time_s"], color="tab:red", alpha=.15, lw=.8)
ax.set(xlabel="time from first recorded CAMERA_HIGH (s)", ylabel="count / confidence", title="Inference coverage on controller time")
ax.legend()
plt.tight_layout()

if not objects.empty:
    display(objects[["detected_this_frame", "tracker_predicted"]].sum().rename({
        "detected_this_frame": "detector observations", "tracker_predicted": "tracker-only predictions"
    }).to_frame("rows"))


## Serial Event Raster


In [ ]:
plot_events = events[~events["eventType"].isin(["CAMERA_HIGH", "CAMERA_LOW"])].dropna(subset=["event_time_s"]).copy()
if plot_events.empty:
    print("No non-camera serial events found.")
else:
    fig, ax = plt.subplots(figsize=(14, max(3, .35 * plot_events["eventType"].nunique() + 2)))
    sns.scatterplot(data=plot_events, x="event_time_s", y="eventType", hue="side" if "side" in plot_events else None, s=65, ax=ax)
    ax.set(xlabel="time from first recorded CAMERA_HIGH (s)", ylabel="event type", title="Non-camera controller events")
    plt.tight_layout()
    display(plot_events["eventType"].value_counts().to_frame("count"))


## Detection-to-Frame Mapping Proof

Live outputs should map by `flir_user_meta`; offline outputs should map by `offline_video_ledger`. Both methods preserve `source_sequence_index`/`raw_frame_num` and then use the dynamic per-run FLIR-to-RP2040 epoch in the aligner.


In [ ]:
if detections.empty:
    print("No detections to audit.")
else:
    method_counts = detections["raw_frame_mapping_method"].fillna("missing").value_counts()
    display(method_counts.to_frame("rows"))
    map_check = detections.merge(
        frames[["camera_frame_id", "frame_pts_ns"]], on="camera_frame_id", how="left",
        suffixes=("_det", "_ledger"),
    )
    map_check["mapping_pts_delta_ns"] = map_check["raw_frame_mapping_pts_ns"] - map_check["frame_pts_ns_ledger"]
    problems = map_check[(map_check["raw_frame_mapping_ok"].fillna(0) != 1) | map_check["camera_frame_id"].isna()]
    print(f"problem mapping rows: {len(problems):,}")
    display(map_check[["detection_index", "raw_frame_index", "camera_frame_id", "raw_frame_mapping_method", "raw_frame_mapping_ok", "mapping_pts_delta_ns"]].head(20))


## NvDCF Track Summary and Bounding-box Trajectories

Use `objects.csv` for trajectories: it contains both current tracker boxes and flags that distinguish detector observations from tracker predictions.


In [ ]:
if objects.empty:
    print("No object rows to plot.")
else:
    objects["cx"] = objects["track_x"] + objects["track_w"] / 2
    objects["cy"] = objects["track_y"] + objects["track_h"] / 2
    display(tracks.sort_values("total_rows", ascending=False).head(20))
    valid = objects.dropna(subset=["track_id", "frame_time_s", "cx", "cy"])
    top_ids = valid["track_id"].value_counts().head(8).index
    plot_objects = valid[valid["track_id"].isin(top_ids)]
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    sns.lineplot(data=plot_objects, x="frame_time_s", y="cx", hue="track_id", estimator=None, ax=axes[0])
    axes[0].set(xlabel="time (s)", ylabel="center x (px)", title="Longest tracks over time")
    sns.scatterplot(data=plot_objects, x="cx", y="cy", hue="track_id", size="detected_this_frame", sizes=(15, 50), ax=axes[1])
    axes[1].invert_yaxis(); axes[1].set_aspect("equal", adjustable="box")
    axes[1].set(xlabel="x (px)", ylabel="y (px)", title="Track paths; larger point = detector observation")
    plt.tight_layout()


## Pose Keypoints

Keypoints are loaded from the normalized `keypoints.csv` table. Names come from the selected model package; no class or pose indices are hard-coded here.


In [ ]:
def selected_pose_sidecar() -> Path:
    if RESULT_DIR != RUN_DIR:
        manifest = json.loads((RESULT_DIR / "offline_manifest.json").read_text())
        return Path(manifest["model_package"]["pose_sidecar"])
    manifest = json.loads((RUN_DIR / "run_manifest.json").read_text())
    return Path(manifest["inference"]["model_package"]["pose_sidecar"])

pose_schema = json.loads(selected_pose_sidecar().read_text())
print(f"model keypoints={pose_schema.get('keypoint_count')}")
if not keypoints.empty:
    display(keypoints.groupby("keypoint_name")["confidence"].describe().sort_values("mean", ascending=False))


In [ ]:
MIN_KEYPOINT_SCORE = float(pose_schema.get("keypoint_threshold", 0.5))
KEYPOINTS_TO_PLOT = list(keypoints["keypoint_name"].dropna().unique()[:6]) if not keypoints.empty else []

pose = keypoints[(keypoints["keypoint_name"].isin(KEYPOINTS_TO_PLOT)) & (keypoints["confidence"] >= MIN_KEYPOINT_SCORE)].copy()
if pose.empty:
    print("No keypoints pass the selected filter.")
else:
    fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
    sns.lineplot(data=pose, x="frame_time_s", y="x_px", hue="keypoint_name", estimator=None, ax=axes[0])
    sns.lineplot(data=pose, x="frame_time_s", y="y_px", hue="keypoint_name", estimator=None, ax=axes[1], legend=False)
    axes[0].set(title="Keypoint x", ylabel="x (px)")
    axes[1].set(title="Keypoint y", xlabel="time (s)", ylabel="y (px)")
    plt.tight_layout()


## Event-aligned Inference


In [ ]:
def event_locked_objects(event_type: str, pre_s=1.0, post_s=2.0) -> pd.DataFrame:
    anchors = events[(events["eventType"] == event_type) & events["event_time_s"].notna()]
    chunks = []
    for anchor_index, event in anchors.iterrows():
        t0 = float(event["event_time_s"])
        window = objects[objects["frame_time_s"].between(t0 - pre_s, t0 + post_s)].copy()
        window["anchor_index"] = anchor_index
        window["time_from_event_s"] = window["frame_time_s"] - t0
        chunks.append(window)
    return pd.concat(chunks, ignore_index=True) if chunks else pd.DataFrame()

EVENT_TYPE = "POKE_START"
locked = event_locked_objects(EVENT_TYPE)
print(f"{EVENT_TYPE} locked object rows: {len(locked):,}")
if not locked.empty:
    sns.scatterplot(data=locked, x="time_from_event_s", y="detector_confidence", hue="track_id", alpha=.6)
    plt.axvline(0, color="black", ls="--", lw=1)
    plt.title(f"Tracked objects around {EVENT_TYPE}")
    plt.tight_layout()


## Optional Exact-frame Raw Video Preview

This preview seeks by the authoritative raw-video frame index, not an approximate timestamp. It overlays tracker boxes and normalized keypoints from the selected result.


In [ ]:
RUN_PREVIEW = True
RAW_FRAME_INDEX_TO_PREVIEW = 12
# RAW_FRAME_INDEX_TO_PREVIEW = int(frames.iloc[len(frames) // 2]["raw_frame_index"]) if len(frames) else 0
KEYPOINT_SCORE_THRESHOLD = float(pose_schema.get("keypoint_threshold", 0.5))

if RUN_PREVIEW:
    frame_match = frames[frames["raw_frame_index"] == RAW_FRAME_INDEX_TO_PREVIEW]
    if frame_match.empty:
        raise ValueError(f"No ledger row for raw_frame_index={RAW_FRAME_INDEX_TO_PREVIEW}")
    frame_row = frame_match.iloc[0]
    video_path = RUN_DIR / "raw.mp4"
    local_index = int(frame_row.get("video_frame_index", frame_row["raw_frame_index"] - frames["raw_frame_index"].min()))
    out_png = RESULT_DIR / f"preview_raw_{RAW_FRAME_INDEX_TO_PREVIEW:06d}.png"
    subprocess.run([
        "ffmpeg", "-y", "-hide_banner", "-loglevel", "error", "-i", str(video_path),
        "-vf", f"select=eq(n\\,{local_index})", "-vsync", "0", "-frames:v", "1", str(out_png),
    ], check=True)

    frame_objects = objects[objects["source_sequence_index"] == RAW_FRAME_INDEX_TO_PREVIEW]
    frame_points = keypoints[(keypoints["source_sequence_index"] == RAW_FRAME_INDEX_TO_PREVIEW) & (keypoints["confidence"] >= KEYPOINT_SCORE_THRESHOLD)]
    image = plt.imread(out_png)
    fig, ax = plt.subplots(figsize=(12, 8)); ax.imshow(image, cmap="gray")
    cmap = plt.get_cmap("tab10")
    for ordinal, (_, obj) in enumerate(frame_objects.iterrows()):
        color = cmap(ordinal % 10)
        ax.add_patch(Rectangle((obj["track_x"], obj["track_y"]), obj["track_w"], obj["track_h"], fill=False, edgecolor=color, lw=2))
        ax.text(obj["track_x"], max(0, obj["track_y"] - 5), f"{obj['class_label']} T{obj['track_id']}", color=color)
        points = frame_points[frame_points["observation_id"] == obj["observation_id"]]
        ax.scatter(points["x_px"], points["y_px"], color=[color], s=24)
    ax.set_title(f"raw_frame_index={RAW_FRAME_INDEX_TO_PREVIEW}; camera_frame_id={int(frame_row['camera_frame_id'])}")
    ax.set_xlim(0, image.shape[1]); ax.set_ylim(image.shape[0], 0); ax.axis("off")
    plt.tight_layout()
    display(frame_objects, frame_points)
else:
    print("Preview disabled. Set RUN_PREVIEW=True and choose RAW_FRAME_INDEX_TO_PREVIEW.")
